# Veda GPU Server on Kaggle

**Dataset:** attach **`project-models`** (latest version).

**Publish updates from laptop** (when code changes):
```bash
server\kaggle\publish_kaggle_dataset.bat
```
Requires `KAGGLE_USERNAME` + `KAGGLE_KEY` in `server/.env`.

**Each session:** GPU T4 x2 · Internet ON · set `KAGGLE_API_SECRET` in cell 2 · copy tunnel URL to laptop `server/.env`.

In [ ]:
# Copy server modules from the attached veda-models dataset into /kaggle/working
import shutil
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

REQUIRED = (
    "veda_gpu_server.py",
    "security.py",
    "qwen_config.py",
    "analyze_prompts.py",
    "page_layout_service.py",
    "image_gen_service.py",
)

missing = []
for name in REQUIRED:
    copied = False
    for src in INPUT.rglob(name):
        shutil.copy2(src, WORK / name)
        print(f"Copied {name} ← {src}")
        copied = True
        break
    if not copied:
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "Missing from dataset: " + ", ".join(missing)
        + "\nRe-run prepare_kaggle_dataset.bat + publish_kaggle_dataset.bat on your laptop,"
        + " attach the new dataset version, then re-run this cell."
    )

print("\nWorking dir:", sorted(p.name for p in WORK.iterdir() if p.is_file()))

In [ ]:
import os

# Quiet startup (default). Set VEDA_QUIET=0 for full pip/HF logs.
os.environ.setdefault("VEDA_QUIET", "1")
os.environ["KAGGLE_API_SECRET"] = "PASTE_YOUR_STRONG_SECRET_HERE"

# Persist HuggingFace cache across re-runs (do not "Restart & Clear All").
os.environ.setdefault("HF_HOME", "/kaggle/working/hf_cache")
os.environ.setdefault("HUGGING_FACE_HUB_CACHE", "/kaggle/working/hf_cache")

# img2img model (default SSD-1B — ~3 GB, faster load than full SDXL).
# os.environ["SDXL_MODEL"] = "stabilityai/stable-diffusion-xl-base-1.0"
# os.environ["SDXL_MODEL"] = "segmind/Segmind-Vega"

!python /kaggle/working/veda_gpu_server.py